# University Rankings 2014–2024 — Unified DataFrame

This notebook loads the yearly ranking files (`2014.csv` ... `2024.csv`) and builds
a single, consistent `pandas.DataFrame` that:

- Keeps **only the columns common to every year**.
- Adds a **`Year`** column, taken from the file name.
- Fixes the **`World Rank`** column: in some years (e.g. 2017, 2023, 2024) this field
  is stored across two lines, e.g.:

  ```
  1
  Top 0.1%    Harvard University  USA ...
  ```

  Here we keep only the numeric rank (`1`) and discard the `Top X%` part.
- Unifies the country column name: some files call it `Location`, others `Country`
  → we standardize on `Location`.

**Note on the file format:** despite the `.csv` extension, these files are **not**
comma-separated — fields are separated by **TAB** characters.


## 1. Imports

In [61]:
import io
import re
from pathlib import Path

import pandas as pd


## 2. Configuration

Point `DATA_DIR` to the folder containing the yearly CSV files
(`2014.csv`, `2015.csv`, ..., `2024.csv`).


In [62]:
DATA_DIR = Path("./ranking")  # <-- change this if your files live elsewhere


## 3. Load a single year file

Steps performed for each file:

1. Read the raw text.
2. Repair rows whose `World Rank` field was split across two physical lines
   (`"<n>\nTop X%\t..."` → `"<n>\t..."`).
3. Parse the repaired text as a TAB-separated table.
4. Rename `Country` → `Location` (only present in some years) for consistency.
5. Force `World Rank` to a clean integer.
6. Add the `Year` column, extracted from the file name.


In [63]:
def load_year_file(path: Path) -> pd.DataFrame:
    """Load and clean a single yearly ranking file."""
    raw_text = path.read_text(encoding="utf-8", errors="replace")

    # A wrapped 'World Rank' field looks like:
    #   1
    #   Top 0.1%<TAB>Harvard University<TAB>...
    # i.e. the numeric rank, a newline, then "Top X%" followed by the rest
    # of the row. We collapse it back into a single row, keeping only the
    # numeric rank and dropping the "Top X%" part entirely.
    wrapped_rank_pattern = re.compile(r"(\d+)\nTop\s+[\d.]+%\t")
    clean_text = wrapped_rank_pattern.sub(r"\1\t", raw_text)

    df = pd.read_csv(io.StringIO(clean_text), sep="\t")

    # Some years call the country column "Country" instead of "Location".
    df = df.rename(columns={"Country": "Location"})

    # World Rank should now be a clean integer in every year.
    df["World Rank"] = (
        df["World Rank"].astype(str).str.extract(r"(\d+)").astype("Int64")
    )

    # Extract the year from the file name (e.g. "2017.csv" -> 2017)
    year = int(re.search(r"(\d{4})", path.stem).group(1))
    df["Year"] = year

    return df


## 4. Build the unified DataFrame

We load every year, compute the set of columns that is present in **all** of them,
and concatenate the yearly tables keeping only those common columns (plus `Year`,
which is always kept).

In [64]:
def build_consistent_dataframe(data_dir: Path) -> pd.DataFrame:
    files = sorted(data_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {data_dir!r}")

    per_year_dfs = [load_year_file(f) for f in files]

    # Columns common to every single year.
    common_cols = set(per_year_dfs[0].columns)
    for df in per_year_dfs[1:]:
        common_cols &= set(df.columns)
    common_cols.discard("Year")  # Year is handled separately, always kept

    # Preserve a sensible column order (order of first file).
    ordered_common_cols = [c for c in per_year_dfs[0].columns if c in common_cols]
    final_cols = ordered_common_cols + ["Year"]

    combined = pd.concat(
        [df[final_cols] for df in per_year_dfs],
        ignore_index=True,
    )

    combined = combined.rename(columns={"Location": "Country"})

    return combined


df = build_consistent_dataframe(DATA_DIR)


In [65]:
df.head(10)

,World Rank,Institution,Country,National Rank,Education Rank,Employability Rank,Faculty Rank,Score,Year
0,1,Harvard University,USA,1,1,1,1,100.00,2014
1,2,Stanford University,USA,2,11,2,4,99.09,2014
2,3,Massachusetts Institute of Technology,USA,3,3,11,2,98.69,2014
3,4,University of Cambridge,United Kingdom,1,2,10,5,97.64,2014
4,5,University of Oxford,United Kingdom,2,7,12,10,97.51,2014
5,6,Columbia University,USA,4,13,8,9,97.41,2014
6,7,"University of California, Berkeley",USA,5,4,22,6,92.84,2014
7,8,University of Chicago,USA,6,10,14,8,92.03,2014
8,9,Princeton University,USA,7,5,16,3,88.56,2014
9,10,Yale University,USA,8,9,25,11,88.11,2014


## 7. Filter: European countries only

We now restrict the dataset to European countries. Turkey is explicitly included
even though it is often treated as transcontinental. Other borderline
transcontinental/post-Soviet countries present in the data (Russia, Belarus,
Armenia, Georgia) are **excluded**, per the "classic Europe" definition used here.

Feel free to edit the `EUROPEAN_COUNTRIES` set below to adjust the list.

In [66]:
EUROPEAN_COUNTRIES = {
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary",
    "Iceland", "Ireland", "Italy", "Lithuania", "Luxembourg", "Netherlands",
    "Norway", "Poland", "Portugal", "Romania", "Serbia", "Slovak Republic",
    "Slovenia", "Spain", "Sweden", "Switzerland", "United Kingdom",
    "Turkey"
}

df_europe = df[df["Country"].isin(EUROPEAN_COUNTRIES)].reset_index(drop=True)


In [67]:
df_europe.head(10)


,World Rank,Institution,Country,National Rank,Education Rank,Employability Rank,Faculty Rank,Score,Year
0,4,University of Cambridge,United Kingdom,1,2,10,5,97.64,2014
1,5,University of Oxford,United Kingdom,2,7,12,10,97.51,2014
2,18,Swiss Federal Institute of Technology in Zurich,Switzerland,1,16,105,13,72.18,2014
3,30,University College London,United Kingdom,3,20,406,52,61.05,2014
4,35,École normale supérieure - Paris,France,1,8,478+,59,59.72,2014
5,36,École Polytechnique,France,2,150,6,208,59.54,2014
6,39,Imperial College London,United Kingdom,4,121,98,38,58.85,2014
7,50,University of Paris-Sud,France,3,26,410,25,56.06,2014
8,56,University of Edinburgh,United Kingdom,5,42,131,36,55.20,2014
9,68,Pierre-and-Marie-Curie University,France,4,36,431,86,53.91,2014


## 8. Normalization

In [68]:
# --- Renormalize rankings on the European (+ Turkish) subset only ---
# Keep the original global rank for reference, then compute a new
# European Rank using ONLY the European+Turkish institutions as the
# reference population (based on Score).

df_europe = df_europe.rename(columns={"World Rank": "Global World Rank"})

df_europe["European Rank"] = (
    df_europe.groupby("Year")["Score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

df_europe["National Rank"] = (
    df_europe.groupby(["Year", "Country"])["Score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

df_europe = df_europe.sort_values(["Year", "European Rank"]).reset_index(drop=True)
df_europe.head(10)

,Global World Rank,Institution,Country,National Rank,Education Rank,Employability Rank,Faculty Rank,Score,Year,European Rank
0,4,University of Cambridge,United Kingdom,1,2,10,5,97.64,2014,1
1,5,University of Oxford,United Kingdom,2,7,12,10,97.51,2014,2
2,18,Swiss Federal Institute of Technology in Zurich,Switzerland,1,16,105,13,72.18,2014,3
3,30,University College London,United Kingdom,3,20,406,52,61.05,2014,4
4,35,École normale supérieure - Paris,France,1,8,478+,59,59.72,2014,5
5,36,École Polytechnique,France,2,150,6,208,59.54,2014,6
6,39,Imperial College London,United Kingdom,4,121,98,38,58.85,2014,7
7,50,University of Paris-Sud,France,3,26,410,25,56.06,2014,8
8,56,University of Edinburgh,United Kingdom,5,42,131,36,55.20,2014,9
9,68,Pierre-and-Marie-Curie University,France,4,36,431,86,53.91,2014,10
